# LeetCode 3574 — Maximize Subarray GCD Score bằng Python

- Link: https://leetcode.com/problems/maximize-subarray-gcd-score/
- Độ khó: **Hard**
- Chủ đề: **GCD của mảng con**, **lũy thừa của 2 trong phân tích thừa số** (`v2`), **duyệt mọi mảng con** với cập nhật tăng dần, tham lam dùng lượt nhân đôi

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxGCDScore(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Tách mỗi số thành `x = odd · 2^v` (phần lẻ và số mũ của 2) bằng thao tác bit `x & -x`.
2. Hiểu `gcd` của một mảng = `gcd(các phần lẻ) · 2^(min v)` ⇒ nhân đôi phần tử **không đổi phần lẻ**, chỉ có thể tăng `min v` thêm **đúng 1**.
3. Suy ra quy tắc tham lam: GCD của mảng con nhân đôi được ⇔ **số phần tử đạt `min v` ≤ k**.
4. Duyệt mọi mảng con trong `O(n^2)` bằng cập nhật tăng dần (GCD, `min v`, số lần đạt `min v`).
5. Kiểm chứng bằng vét cạn mọi tập phần tử được nhân đôi trên mảng nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` và số `k`. Được thực hiện **tối đa `k` lần**: chọn một phần tử và **nhân đôi** nó; mỗi phần tử nhân đôi **tối đa một lần**.

**Điểm** của một mảng con liên tiếp = `độ dài × GCD các phần tử`. Trả về điểm lớn nhất có thể (chọn cách nhân đôi và mảng con tốt nhất).

**Ví dụ 1:** `nums = [2,4], k = 1` → `8` (nhân đôi 2 thành 4: `[4,4]`, `2 × 4`).

**Ví dụ 2:** `nums = [3,5,7], k = 2` → `14` (nhân đôi 7: `[14]`, `1 × 14`).

**Ví dụ 3:** `nums = [5,5,5], k = 1` → `15`.

**Ràng buộc:** `1 ≤ n ≤ 1500`, `1 ≤ nums[i] ≤ 10^9`, `1 ≤ k ≤ n`.

In [1]:
from typing import List
from math import gcd
from functools import reduce
examples = [([2, 4], 1, 8), ([3, 5, 7], 2, 14), ([5, 5, 5], 1, 15)]
for nums, k, exp in examples:
    print(nums, k, '-> kỳ vọng', exp)

[2, 4] 1 -> kỳ vọng 8
[3, 5, 7] 2 -> kỳ vọng 14
[5, 5, 5] 1 -> kỳ vọng 15


## 3. Kiến thức nền

### 3.1. GCD dồn của mảng con

`gcd(a, b, c) = gcd(gcd(a, b), c)`. Khi mở rộng mảng con sang phải, GCD **giảm hoặc giữ nguyên** và chỉ đổi giá trị `O(log V)` lần.

In [2]:
print(reduce(gcd, [12, 18, 24]))
g = 0
for x in [48, 36, 30, 7]:
    g = gcd(g, x)
    print('thêm', x, '-> gcd =', g)

6
thêm 48 -> gcd = 48
thêm 36 -> gcd = 12
thêm 30 -> gcd = 6
thêm 7 -> gcd = 1


### 3.2. Số mũ của 2: `v2(x)` và phần lẻ

`x & -x` cho lũy thừa của 2 lớn nhất chia hết `x` (bit thấp nhất). Vậy `v2(x) = (x & -x).bit_length() - 1` và phần lẻ `odd(x) = x >> v2(x)`.

In [3]:
def split2(x):
    v = (x & -x).bit_length() - 1
    return x >> v, v

for x in [12, 7, 96, 1 << 29]:
    o, v = split2(x)
    print(f"{x} = {o} · 2^{v}")

12 = 3 · 2^2
7 = 7 · 2^0
96 = 3 · 2^5
536870912 = 1 · 2^29


### 3.3. GCD qua phần lẻ và `min v2`

Với mảng `a`: `gcd(a) = gcd(odd(a_i)) · 2^(min v2(a_i))` — vì thừa số 2 chung chính là `2^min`, còn phần lẻ không chứa thừa số 2.

In [4]:
a = [12, 40, 28]
odd_g = reduce(gcd, [split2(x)[0] for x in a])
mv = min(split2(x)[1] for x in a)
print('gcd trực tiếp =', reduce(gcd, a), '| qua công thức =', odd_g << mv)

gcd trực tiếp = 4 | qua công thức = 4


### 3.4. Nhân đôi tác động thế nào?

Nhân đôi `x` ⇒ `odd` giữ nguyên, `v2` tăng 1. Vậy trong một mảng con:

- `gcd(odd)` **không đổi** dù nhân đôi gì.
- `min v2` chỉ tăng khi **mọi** phần tử đang đạt `min v2` đều được nhân đôi; khi đó tăng đúng 1 (mỗi phần tử nhân đôi tối đa một lần).
⇒ GCD mảng con sau khi nhân đôi tối ưu bằng `gcd` gốc **× 2** nếu `cnt_min ≤ k`, ngược lại giữ nguyên.

In [5]:
a = [12, 40, 28]           # v2 = 2, 3, 2 -> có 2 phần tử đạt min
for k in [1, 2]:
    vs = [split2(x)[1] for x in a]
    cnt_min = vs.count(min(vs))
    print(f"k={k}: cnt_min={cnt_min} -> gcd tốt nhất = {reduce(gcd, a) * (2 if cnt_min <= k else 1)}")

k=1: cnt_min=2 -> gcd tốt nhất = 4
k=2: cnt_min=2 -> gcd tốt nhất = 8


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ta muốn **tối đa** `độ dài × GCD`. Hai cách tăng điểm: chọn đoạn dài (GCD thường giảm) hoặc nhân đôi để GCD gấp đôi.

Tính tay ví dụ 1: `[2, 4]`, `k = 1`. `2 = 1·2^1`, `4 = 1·2^2`.

| mảng con | gcd(odd) | min v2 | cnt_min | gcd sau nhân đôi | điểm |
|---|---|---|---|---|---|
| [2] | 1 | 1 | 1 ≤ 1 | 4 | 4 |
| [4] | 1 | 2 | 1 ≤ 1 | 8 | 8 |
| [2,4] | 1 | 1 | 1 ≤ 1 | 4 | **8** |

Đáp án `8` (đạt ở hai cách). Chú ý **lượt nhân đôi được dùng riêng cho mảng con được chọn** — vì ta chỉ chấm điểm một mảng con, nên dùng hết lượt cho nó.

In [6]:
def show_table(nums, k):
    n = len(nums)
    for l in range(n):
        for r in range(l, n):
            sub = nums[l:r + 1]
            vs = [split2(x)[1] for x in sub]
            g = reduce(gcd, sub)
            c = vs.count(min(vs))
            gg = g * (2 if c <= k else 1)
            print(f"{sub}: gcd={g}, cnt_min={c} -> gcd tối ưu {gg}, điểm {len(sub) * gg}")

show_table([2, 4], 1)

[2]: gcd=2, cnt_min=1 -> gcd tối ưu 4, điểm 4
[2, 4]: gcd=2, cnt_min=1 -> gcd tối ưu 4, điểm 8
[4]: gcd=4, cnt_min=1 -> gcd tối ưu 8, điểm 8


## 5. Cách 1 — Brute Force: thử mọi tập phần tử nhân đôi

Thử mọi tập con các chỉ số kích thước `≤ k`, nhân đôi chúng, rồi duyệt mọi mảng con tính `độ dài × gcd`. Không dùng bất kỳ suy luận nào ⇒ dùng làm oracle.

In [7]:
from itertools import combinations

def max_score_brute(nums, k):
    n = len(nums)
    best = 0
    for t in range(0, min(k, n) + 1):
        for pos in combinations(range(n), t):
            b = nums[:]
            for p in pos:
                b[p] *= 2
            for l in range(n):
                g = 0
                for r in range(l, n):
                    g = gcd(g, b[r])
                    best = max(best, (r - l + 1) * g)
    return best

for nums, k, exp in examples:
    print(max_score_brute(nums, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
14 | kỳ vọng 14
15 | kỳ vọng 15


**Độ phức tạp:** `O(C(n, ≤k) · n^2)` — mũ.

**Nút thắt:** việc chọn tập nhân đôi. Theo mục 3.4, tập tối ưu **cho một mảng con cố định** được xác định ngay: nhân đôi các phần tử đạt `min v2` (nếu đủ lượt).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** cố định mảng con `[l, r]`. Gọi `G = gcd(nums[l..r])`, `m = min v2`, `c` = số phần tử có `v2 = m`. Điểm tối ưu của mảng con này là:

`(r − l + 1) · G · (2 nếu c ≤ k, ngược lại 1)`.

Vì các mảng con khác nhau không ảnh hưởng lẫn nhau (ta chỉ chấm một mảng con), đáp án = max trên mọi mảng con.

**Cập nhật tăng dần:** cố định `l`, tăng `r`:

- `g = gcd(g, nums[r])` (hoặc gcd phần lẻ rồi nhân `2^m`);
- nếu `v2[r] < m`: `m = v2[r]`, `c = 1`; nếu bằng: `c += 1`.

⇒ `O(n^2)` mảng con, mỗi mảng con `O(1)` (cộng chi phí `gcd`).

**Cắt tỉa (tùy chọn):** với `l` cố định, điểm tối đa có thể còn đạt là `(n − l) · g · 2`; nếu `≤ best` thì dừng sớm vì `g` chỉ giảm.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mọi mảng con, tính lại từ đầu (O(n^3))

In [8]:
def max_score_cubic(nums, k):
    n = len(nums)
    best = 0
    for l in range(n):
        for r in range(l, n):
            sub = nums[l:r + 1]
            g = reduce(gcd, sub)
            vs = [split2(x)[1] for x in sub]
            c = vs.count(min(vs))
            best = max(best, len(sub) * g * (2 if c <= k else 1))
    return best

for nums, k, exp in examples:
    print(max_score_cubic(nums, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
14 | kỳ vọng 14
15 | kỳ vọng 15


**Độ phức tạp:** `O(n^3)` ≈ `5·10^8` với `n = 1500` — quá chậm.

### 7.2. Cách 3 — Cập nhật tăng dần (O(n^2), lời giải chính)

In [9]:
def max_score_quadratic(nums, k):
    n = len(nums)
    odd, v2 = [], []
    for x in nums:
        v = (x & -x).bit_length() - 1
        v2.append(v)
        odd.append(x >> v)
    best = 0
    for l in range(n):
        g, m, c = 0, 64, 0
        for r in range(l, n):
            g = gcd(g, odd[r])
            if v2[r] < m:
                m, c = v2[r], 1
            elif v2[r] == m:
                c += 1
            s = (r - l + 1) * (g << m)
            if c <= k:
                s <<= 1
            if s > best:
                best = s
    return best

for nums, k, exp in examples:
    print(max_score_quadratic(nums, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
14 | kỳ vọng 14
15 | kỳ vọng 15


**Độ phức tạp:** `O(n^2 · log V)` (chi phí `gcd`), Space `O(n)`.

### 7.3. Cách 4 — O(n^2) + cắt tỉa theo cận trên

Vì `g` chỉ giảm khi `r` tăng, điểm của mọi mảng con `[l, r']` với `r' ≥ r` không vượt `(n − l) · (g << m_cur) · 2`... Lưu ý `m` cũng chỉ giảm. Ta dùng cận `(n − l) · G_cur · 2` với `G_cur = g << m`: nếu `≤ best` thì `break`.

In [10]:
def max_score_pruned(nums, k):
    n = len(nums)
    odd, v2 = [], []
    for x in nums:
        v = (x & -x).bit_length() - 1
        v2.append(v)
        odd.append(x >> v)
    best = 0
    for l in range(n):
        if (n - l) * (nums[l] << 1) <= best:     # cận trên ngay từ đầu
            continue
        g, m, c = 0, 64, 0
        for r in range(l, n):
            g = gcd(g, odd[r])
            if v2[r] < m:
                m, c = v2[r], 1
            elif v2[r] == m:
                c += 1
            G = g << m
            s = (r - l + 1) * G
            if c <= k:
                s <<= 1
            if s > best:
                best = s
            if (n - l) * (G << 1) <= best:      # không thể tốt hơn nữa
                break
    return best

for nums, k, exp in examples:
    print(max_score_pruned(nums, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
14 | kỳ vọng 14
15 | kỳ vọng 15


Cận `(n − l) · 2G` đúng vì với mọi `r' ≥ r`: độ dài `≤ n − l`, GCD gốc `≤ G` (giảm dần), nhân đôi tối đa `×2`.

**Độ phức tạp:** xấu nhất vẫn `O(n^2 log V)` nhưng thường nhanh hơn nhiều trên dữ liệu ngẫu nhiên.

### 7.4. Cách 5 — Theo nhóm GCD (ý tưởng `O(n log V)` cho phần GCD)

Một góc nhìn nâng cao: cố định `r`, các giá trị `gcd(nums[l..r])` khi `l` giảm chỉ có `O(log V)` giá trị khác nhau. Với mỗi nhóm, độ dài lớn nhất là tốt nhất **nếu không xét nhân đôi**. Phần nhân đôi phụ thuộc `cnt_min`, phức tạp hơn để gộp nhóm — với `n ≤ 1500`, cách 3/4 là đủ. Dưới đây là phiên bản "không nhân đôi" để minh họa danh sách GCD:

In [11]:
def max_score_no_double(nums):
    best, cur = 0, []           # cur: danh sách (gcd, l nhỏ nhất)
    for r, x in enumerate(nums):
        nxt = [(x, r)]
        for g, l in cur:
            ng = gcd(g, x)
            if ng == nxt[-1][0]:
                nxt[-1] = (ng, l)
            else:
                nxt.append((ng, l))
        cur = nxt
        for g, l in cur:
            best = max(best, (r - l + 1) * g)
    return best

print(max_score_no_double([5, 5, 5]), max_score_no_double([2, 4]), max_score_no_double([3, 5, 7]))

15 4 7


## 8. TRACE từng bước

Trace cách 3 trên `nums = [12, 40, 28, 6]`, `k = 1`: in `(g_lẻ, m, c)` và điểm từng mảng con.

In [12]:
def trace_quadratic(nums, k):
    n = len(nums)
    sp = [split2(x) for x in nums]
    print('phần lẻ/v2:', sp)
    best, arg = 0, None
    for l in range(n):
        g, m, c = 0, 64, 0
        for r in range(l, n):
            o, v = sp[r]
            g = gcd(g, o)
            if v < m:
                m, c = v, 1
            elif v == m:
                c += 1
            G = g << m
            s = (r - l + 1) * G * (2 if c <= k else 1)
            mark = ''
            if s > best:
                best, arg = s, (l, r)
                mark = '  <- tốt nhất mới'
            print(f"[{l},{r}] g_lẻ={g:>2} m={m} c={c} gcd={G:>3} {'×2' if c <= k else '  '} điểm={s:>4}{mark}")
    print('Đáp án =', best, 'tại mảng con', arg)
    return best

trace_quadratic([12, 40, 28, 6], 1)

phần lẻ/v2: [(3, 2), (5, 3), (7, 2), (3, 1)]
[0,0] g_lẻ= 3 m=2 c=1 gcd= 12 ×2 điểm=  24  <- tốt nhất mới
[0,1] g_lẻ= 1 m=2 c=1 gcd=  4 ×2 điểm=  16
[0,2] g_lẻ= 1 m=2 c=2 gcd=  4    điểm=  12
[0,3] g_lẻ= 1 m=1 c=1 gcd=  2 ×2 điểm=  16
[1,1] g_lẻ= 5 m=3 c=1 gcd= 40 ×2 điểm=  80  <- tốt nhất mới
[1,2] g_lẻ= 1 m=2 c=1 gcd=  4 ×2 điểm=  16
[1,3] g_lẻ= 1 m=1 c=1 gcd=  2 ×2 điểm=  12
[2,2] g_lẻ= 7 m=2 c=1 gcd= 28 ×2 điểm=  56
[2,3] g_lẻ= 1 m=1 c=1 gcd=  2 ×2 điểm=   8
[3,3] g_lẻ= 3 m=1 c=1 gcd=  6 ×2 điểm=  12
Đáp án = 80 tại mảng con (1, 1)


80

Đọc trace: mảng con `[12, 40, 28]` có `gcd = 4`, hai phần tử (12, 28) đạt `v2 = 2` nhưng `k = 1` ⇒ không nhân đôi được. `[40]` một mình với nhân đôi cho `80`.

In [13]:
print('vét cạn:', max_score_brute([12, 40, 28, 6], 1))

vét cạn: 80


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxGCDScore(self, nums: List[int], k: int) -> int:
        n = len(nums)
        odd, v2 = [0] * n, [0] * n
        for i, x in enumerate(nums):
            v = (x & -x).bit_length() - 1
            v2[i], odd[i] = v, x >> v
        best = 0
        for l in range(n):
            if (n - l) * (nums[l] << 1) <= best:
                continue
            g, m, c = 0, 64, 0
            for r in range(l, n):
                g = gcd(g, odd[r])
                vr = v2[r]
                if vr < m:
                    m, c = vr, 1
                elif vr == m:
                    c += 1
                G = g << m
                s = (r - l + 1) * G
                if c <= k:
                    s <<= 1
                if s > best:
                    best = s
                if (n - l) * (G << 1) <= best:
                    break
        return best

for nums, k, exp in examples:
    got = Solution().maxGCDScore(nums, k)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
14 OK
15 OK


## 10. Vì sao thuật toán đúng

1. **Chỉ một mảng con được chấm điểm** ⇒ với mảng con tối ưu, có thể dồn toàn bộ lượt nhân đôi vào nó; nhân đôi ngoài mảng con vô ích.
2. **Phân tích GCD:** `gcd(a) = gcd(odd) · 2^(min v2)`. Nhân đôi không đổi `odd` ⇒ `gcd(odd)` cố định.
3. **Tăng `min v2`:** sau nhân đôi, `v2` mỗi phần tử tăng tối đa 1 ⇒ `min v2` tăng tối đa 1, và tăng ⇔ mọi phần tử đạt min đều được nhân đôi ⇔ cần `c` lượt. Vậy GCD tối ưu = `G · 2` nếu `c ≤ k`, ngược lại `G` (nhân đôi phần tử khác không làm GCD tăng).
4. **Cập nhật tăng dần đúng:** `g`, `m`, `c` sau bước `r` đúng bằng các đại lượng của `[l, r]` theo định nghĩa.
5. **Cắt tỉa đúng:** với `r' ≥ r`, `gcd(odd)` và `m` đều không tăng nên `G' ≤ G`; điểm `≤ (n − l) · 2G ≤ best` ⇒ bỏ qua an toàn. Tương tự với `continue` ban đầu (`G ≤ nums[l]`).

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 1500`, `V = 10^9`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập nhân đôi | `O(C(n,≤k) · n^2 log V)` | `O(n)` |
| 2. Mọi mảng con, tính lại | `O(n^3)` | `O(n)` |
| 3. Cập nhật tăng dần | `O(n^2 log V)` | `O(n)` |
| 4. + cắt tỉa | xấu nhất `O(n^2 log V)`, thường nhanh hơn | `O(n)` |

`n^2/2 ≈ 1.1·10^6` mảng con — Python xử lý trong khoảng 0,5–1 s.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [15]:
impls = {
    'cubic': max_score_cubic,
    'quadratic': max_score_quadratic,
    'pruned': max_score_pruned,
    'submit': lambda a, k: Solution().maxGCDScore(a, k),
}
edge = [
    ([1], 1, 2),                     # nhân đôi 1 -> 2
    ([10**9], 1, 2 * 10**9),
    ([1, 1, 1, 1], 2, 4),            # cả mảng gcd 1 (×2 cần 4 lượt) vs [1,1] ×2 = 4
    ([1, 1, 1, 1], 4, 8),
    ([6, 12, 24], 1, 24 * 2),        # [24] nhân đôi = 48; [6,12,24]: gcd 6, cnt_min=1 -> 36
    ([7, 7, 14, 7], 3, 4 * 14),      # cả mảng: gcd 7, 3 phần tử v2=0 -> ×2
]
for nums, k, exp in examples + edge:
    for name, f in impls.items():
        got = f(nums, k)
        assert got == exp, (name, nums, k, got, exp)
    assert max_score_brute(nums, k) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [16]:
import random
random.seed(3574)
cnt = 0
for _ in range(500):
    n = random.randint(1, 7)
    nums = [random.choice([random.randint(1, 20), 2 ** random.randint(0, 5) * random.choice([1, 3, 5])])
            for _ in range(n)]
    k = random.randint(1, n)
    exp = max_score_brute(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 500 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 1500`)

In [17]:
import time
random.seed(15)
n = 1500
cases = {
    'ngẫu nhiên lớn': [random.randint(1, 10**9) for _ in range(n)],
    'bội của 6 · 2^t': [6 * 2 ** random.randint(0, 20) for _ in range(n)],
    'toàn bằng nhau': [7] * n,
    'lũy thừa của 2': [2 ** random.randint(0, 29) for _ in range(n)],
}
for name, nums in cases.items():
    k = random.randint(1, n)
    t = time.time(); a = Solution().maxGCDScore(nums, k); t1 = time.time() - t
    t = time.time(); b = max_score_quadratic(nums, k); t2 = time.time() - t
    assert a == b
    print(f"{name:>16}: đáp án {a} | có cắt tỉa {t1:.2f}s | không cắt tỉa {t2:.2f}s")

  ngẫu nhiên lớn: đáp án 1999985204 | có cắt tỉa 0.00s | không cắt tỉa 0.20s


 bội của 6 · 2^t: đáp án 37748736 | có cắt tỉa 0.00s | không cắt tỉa 0.18s


  toàn bằng nhau: đáp án 10794 | có cắt tỉa 0.18s | không cắt tỉa 0.19s
  lũy thừa của 2: đáp án 2147483648 | có cắt tỉa 0.00s | không cắt tỉa 0.20s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhân đôi luôn được ×2 nếu còn lượt (quên điều kiện `cnt_min ≤ k`)

In [18]:
def wrong_always_double(nums, k):
    n, best = len(nums), 0
    for l in range(n):
        g = 0
        for r in range(l, n):
            g = gcd(g, nums[r])
            best = max(best, (r - l + 1) * g * 2)      # SAI
    return best

print('sai :', wrong_always_double([5, 5, 5], 1))  # 30
print('đúng:', Solution().maxGCDScore([5, 5, 5], 1))  # 15

sai : 30
đúng: 15


**Vì sao sai:** muốn GCD gấp đôi thì **mọi** phần tử đạt `min v2` phải được nhân đôi; với `[5,5,5]` cần 3 lượt.

### Lỗi 2 — Đếm số phần tử **lẻ** thay vì số phần tử đạt `min v2`

In [19]:
def wrong_count_odd(nums, k):
    n, best = len(nums), 0
    for l in range(n):
        g, c = 0, 0
        for r in range(l, n):
            g = gcd(g, nums[r])
            c += nums[r] % 2                          # SAI: chỉ đếm số lẻ
            best = max(best, (r - l + 1) * g * (2 if c <= k else 1))
    return best

print('sai :', wrong_count_odd([4, 8, 4], 1))     # coi c = 0 -> ×2 cả mảng
print('đúng:', Solution().maxGCDScore([4, 8, 4], 1))

sai : 24
đúng: 16


**Vì sao sai:** `[4, 8, 4]` không có số lẻ nhưng `min v2 = 2` đạt ở **hai** phần tử (4 và 4) ⇒ một lượt không đủ.

### Lỗi 3 — Nghĩ rằng nhân đôi có thể tăng GCD **nhiều hơn** 2 lần

In [20]:
def wrong_multi(nums, k):
    # SAI: lặp "nhân đôi các phần tử min" nhiều vòng như thể mỗi phần tử nhân đôi nhiều lần
    n, best = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            vs = sorted(split2(x)[1] for x in nums[l:r + 1])
            g = reduce(gcd, nums[l:r + 1])
            used, factor, i = 0, 1, 0
            m = vs[0]
            while True:
                cnt = sum(1 for v in vs if v == m)
                if used + cnt > k:
                    break
                used += cnt
                factor *= 2
                vs = [v + 1 if v == m else v for v in vs]
                m += 1
            best = max(best, (r - l + 1) * g * factor)
    return best

print('sai :', wrong_multi([3, 5, 7], 3))   # "nhân đôi" 7 ba lần -> 56
print('đúng:', Solution().maxGCDScore([3, 5, 7], 3))  # 14

sai : 56
đúng: 14


**Vì sao sai:** mỗi phần tử nhân đôi **tối đa một lần** ⇒ `min v2` tăng tối đa 1.

### Lỗi 4 — Tính `v2` bằng vòng `while x % 2 == 0` nhưng quên dùng biến tạm (làm hỏng mảng gốc)

In [21]:
def v2_destroy(nums):
    vs = []
    for i in range(len(nums)):
        v = 0
        while nums[i] % 2 == 0:        # SAI: sửa trực tiếp nums[i]
            nums[i] //= 2
            v += 1
        vs.append(v)
    return vs

a = [12, 40]
print('v2:', v2_destroy(a), '| mảng sau khi tính:', a, '-> gcd sai nếu dùng tiếp a')
b = [12, 40]
print('cách an toàn:', [split2(x) for x in b], '| mảng giữ nguyên:', b)

v2: [2, 3] | mảng sau khi tính: [3, 5] -> gcd sai nếu dùng tiếp a
cách an toàn: [(3, 2), (5, 3)] | mảng giữ nguyên: [12, 40]


**Vì sao sai:** sau khi chạy, `nums` chỉ còn phần lẻ; nếu sau đó tính `gcd(nums)` sẽ thiếu thừa số 2. Lưu `odd` và `v2` ra mảng riêng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhân đôi / nhân 2" kết hợp với **GCD** ⇒ tách **thừa số 2** ra riêng: `x = odd · 2^v`.
- "Mỗi phần tử tối đa một lần" + "tối đa hóa min (của v2)" ⇒ chỉ tăng được 1 bậc, cần đủ lượt cho **mọi** phần tử đạt min.
- `n ≤ 2000` và đại lượng trên mảng con cập nhật được tăng dần ⇒ `O(n^2)` là đủ.
- Đại lượng giảm dần khi mở rộng (GCD) ⇒ có thể **cắt tỉa** bằng cận trên.

## 15. Biến thể của bài

1. Mỗi phần tử được nhân đôi **nhiều lần** (tổng tối đa `k`) ⇒ `min v2` tăng nhiều bậc; cần đếm theo từng mức `v2`.
2. Thao tác là **nhân 3** ⇒ tách thừa số 3 (`v3`) thay cho `v2`.
3. Điểm = `GCD` (không nhân độ dài) ⇒ chỉ cần xét mảng con dài 1.
4. Bài liên quan: LC 2447 *Number of Subarrays With GCD Equal to K*, LC 1250 *Check If It Is a Good Array*.

In [22]:
# Biến thể 2: thao tác nhân 3
def split3(x):
    v = 0
    while x % 3 == 0:
        x //= 3
        v += 1
    return x, v

def max_score_triple(nums, k):
    n, best = len(nums), 0
    sp = [split3(x) for x in nums]
    for l in range(n):
        g, m, c = 0, 64, 0
        for r in range(l, n):
            g = gcd(g, sp[r][0])
            if sp[r][1] < m:
                m, c = sp[r][1], 1
            elif sp[r][1] == m:
                c += 1
            best = max(best, (r - l + 1) * g * 3 ** m * (3 if c <= k else 1))
    return best

print(max_score_triple([3, 9, 6], 1), max_score_triple([5, 5], 2))

27 30


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Thử mọi tập nhân đôi | mũ | `O(n)` | Oracle |
| 2 | Mọi mảng con, tính lại | `O(n^3)` | `O(n)` | Hiểu công thức |
| 3 | Cập nhật tăng dần | `O(n^2 log V)` | `O(n)` | Lời giải chuẩn |
| 4 | + cắt tỉa | thường nhanh hơn | `O(n)` | **Nộp bài** |
| 5 | Danh sách GCD | `O(n log V)` (không nhân đôi) | `O(log V)` | Mở rộng / bài liên quan |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maxGCDScore([8, 8], 1)` = ?

**Bài 2 (tính tay):** `maxGCDScore([6, 10, 14], 3)` = ?

**Bài 3 (điền TODO):** viết `v2(x)` không dùng vòng lặp.

**Bài 4 (tự viết lại):** viết hàm trả về **cặp `(l, r)`** và **tập chỉ số cần nhân đôi** cho đáp án tối ưu.

**Bài 5 (mini challenge):** biến thể 1 — mỗi phần tử được nhân đôi nhiều lần, tổng số lần ≤ k. Viết lời giải `O(n^2 · 30)` và kiểm chứng bằng vét cạn nhỏ.

In [23]:
def v2_todo(x):
    # TODO: dùng x & -x
    raise NotImplementedError

def best_choice_todo(nums, k):
    # TODO: lưu (l, r) tốt nhất và chỉ số đạt min v2 nếu nhân đôi
    raise NotImplementedError

def multi_double_todo(nums, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `8 = 1·2^3`, hai phần tử cùng `v2`. (2) Cả mảng cần 2 lượt; `[8]` nhân đôi = 16. (3) `max(2·8, 16) = 16`.

**Bài 2.** (1) `gcd = 2`, mọi phần tử `v2 = 1`. (2) `k = 3` đủ nhân đôi cả ba ⇒ gcd 4, điểm 12. (3) So với `[14]` nhân đôi = 28 ⇒ đáp án 28.

**Bài 3.** (1) `x & -x` là `2^v`. (2) `.bit_length() − 1`. (3) Kiểm tra với vòng lặp chia 2.

**Bài 4.** (1) Khi cập nhật `best`, lưu `(l, r, c ≤ k)`. (2) Chỉ số cần nhân đôi = các `i` trong `[l, r]` có `v2[i] = m`. (3) Mô phỏng lại để kiểm tra.

**Bài 5.** (1) Với mảng con cố định, thử mức tăng `t = 0, 1, 2, ...`: chi phí `Σ max(0, m + t − v2[i])`. (2) Lấy `t` lớn nhất có chi phí ≤ k. (3) Vét cạn phân bổ lượt trên mảng nhỏ.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
a1 = Solution().maxGCDScore([8, 8], 1)
a2 = Solution().maxGCDScore([6, 10, 14], 3)
assert a1 == 16 == max_score_brute([8, 8], 1)
assert a2 == 28 == max_score_brute([6, 10, 14], 3)

def v2_ans(x):
    return (x & -x).bit_length() - 1

for x in range(1, 5000):
    y, v = x, 0
    while y % 2 == 0:
        y //= 2; v += 1
    assert v2_ans(x) == v
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: v2 đúng')

Bài 1: 16 | Bài 2: 28 | Bài 3: v2 đúng


In [25]:
# Bài 4
def best_choice(nums, k):
    n = len(nums)
    sp = [split2(x) for x in nums]
    best, info = 0, None
    for l in range(n):
        g, m, c = 0, 64, 0
        for r in range(l, n):
            g = gcd(g, sp[r][0])
            if sp[r][1] < m:
                m, c = sp[r][1], 1
            elif sp[r][1] == m:
                c += 1
            s = (r - l + 1) * (g << m) * (2 if c <= k else 1)
            if s > best:
                dbl = [i for i in range(l, r + 1) if sp[i][1] == m] if c <= k else []
                best, info = s, (l, r, dbl)
    return best, info

random.seed(8)
for _ in range(300):
    nums = [random.randint(1, 40) for _ in range(random.randint(1, 8))]
    k = random.randint(1, len(nums))
    best, (l, r, dbl) = best_choice(nums, k)
    b = nums[:]
    for i in dbl:
        b[i] *= 2
    assert len(dbl) <= k and (r - l + 1) * reduce(gcd, b[l:r + 1]) == best == max_score_brute(nums, k)
print('Bài 4: ví dụ', best_choice([12, 40, 28, 6], 1))

Bài 4: ví dụ (80, (1, 1, [1]))


In [26]:
# Bài 5: mỗi phần tử nhân đôi nhiều lần, tổng <= k
def multi_double(nums, k):
    n, best = len(nums), 0
    sp = [split2(x) for x in nums]
    for l in range(n):
        g = 0
        vs = []
        for r in range(l, n):
            g = gcd(g, sp[r][0])
            vs.append(sp[r][1])
            m = min(vs)
            t = 0
            while True:
                cost = sum(max(0, m + t + 1 - v) for v in vs)
                if cost > k:
                    break
                t += 1
            best = max(best, (r - l + 1) * (g << (m + t)))
    return best

def multi_brute(nums, k):
    n, best = len(nums), 0
    def rec(i, left, arr):
        nonlocal best
        if i == n:
            for l in range(n):
                g = 0
                for r in range(l, n):
                    g = gcd(g, arr[r])
                    best = max(best, (r - l + 1) * g)
            return
        for t in range(left + 1):
            rec(i + 1, left - t, arr + [nums[i] << t])
    rec(0, k, [])
    return best

random.seed(9)
for _ in range(150):
    nums = [random.randint(1, 12) for _ in range(random.randint(1, 4))]
    k = random.randint(0, 4)
    assert multi_double(nums, k) == multi_brute(nums, k)
print('Bài 5: ví dụ', multi_double([3, 5, 7], 3), multi_double([2, 4], 3))

Bài 5: ví dụ 56 32


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
v2[i] = (x & -x).bit_length() - 1; odd[i] = x >> v2[i]
for l: g = 0; m = inf; c = 0
    for r >= l:
        g = gcd(g, odd[r]); cập nhật (m, c) theo v2[r]
        score = (r-l+1) * (g << m) * (2 if c <= k else 1)
        (tùy chọn) break nếu (n-l) * 2 * (g << m) <= best
```

**Checklist:**

- [ ] Tách thừa số 2: phần lẻ không đổi khi nhân đôi?
- [ ] Nhân đôi chỉ ×2 khi `cnt_min ≤ k`?
- [ ] Đếm phần tử đạt **min v2**, không phải số phần tử lẻ?
- [ ] Không làm hỏng mảng gốc khi tính `v2`?
- [ ] Cắt tỉa dùng cận đúng (`G` không tăng)?